# 01 — Generate the synthetic claims dataset

Runs all three stages of dataset construction (Section 3.2.1) on a Colab GPU and saves
everything to Google Drive:

| Stage | Module | Output (in Drive) |
|---|---|---|
| A. Structured claims | `src.data.generator` | `data/raw/<version>/base_claims.parquet`, `note_requests.jsonl` |
| B. LLM clinical notes | `src.data.note_synthesis` | `data/raw/<version>/notes.jsonl`, `boilerplate_notes.jsonl` |
| C. Injection, labels, splits | `src.data.build_dataset` | `data/processed/<version>/*.parquet`, `manifest.json` |

**Before running:** Runtime → Change runtime type → T4 GPU. Put a `.env` containing
`GITHUB_TOKEN=...` (a fine-grained, read-only token for `victorgpt0/claimguard`) at
`MyDrive/claimguard/.env`, or add it as a Colab secret; never paste it into a cell.

Stage B appends notes in batches, so if the runtime disconnects, rerun the notebook and it
resumes where it stopped.

In [1]:
DATASET_VERSION = 'v0.4-5k'   # bump when the generator, catalogue or parameters change
N_CLAIMS = 5000
SEED = 42
BRANCH = 'dev'

## 1. Bootstrap: Drive, repository, run record

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO_DIR = '/content/claimguard'

DRIVE_ENV = '/content/drive/MyDrive/claimguard/.env'

def load_env_file(path):
    # Minimal .env reader (KEY=value lines), so this cell needs no extra package.
    if not os.path.exists(path):
        return
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"\''))

def github_token():
    # The kernel runs on Colab's VM, so a local .env (VS Code's jupyter.envFile) never
    # reaches it. Read the .env kept in Drive, then a Colab secret, then prompt.
    load_env_file(DRIVE_ENV)
    if os.getenv('GITHUB_TOKEN'):
        return os.environ['GITHUB_TOKEN']
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    from getpass import getpass
    return getpass('GITHUB_TOKEN (read-only, not saved): ').strip()

if not os.path.exists(REPO_DIR):
    token = github_token()
    if not token:
        raise RuntimeError('No GITHUB_TOKEN: add a Colab secret or enter it at the prompt')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    f'https://{token}@github.com/victorgpt0/claimguard.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
ROOT = '/content/drive/MyDrive/claimguard'
RAW = f'{ROOT}/data/raw/{DATASET_VERSION}'
PROCESSED = f'{ROOT}/data/processed/{DATASET_VERSION}'
ENV_LOGS = f'{ROOT}/artifacts/environment_logs'
for d in (RAW, PROCESSED, ENV_LOGS):
    os.makedirs(d, exist_ok=True)
print('commit', COMMIT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
commit fe2ad3a127fc58ffdaab53f04e08d33b70a6d7d0


In [3]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Python 3.13.15


## 2. Stage A: structured claims (CPU)

In [4]:
!pip install -q pyarrow pyyaml
!python -m src.data.generator --n-claims {N_CLAIMS} --seed {SEED} --out {RAW}

5000 claims, 60 providers -> /content/drive/MyDrive/claimguard/data/raw/v0.4-5k
  inflated caesarean claims: 81
  fictitious dependent claims: 83


## 3. Stage B: clinical notes (GPU)

Tries vLLM first (pinned to a release built for Colab's CUDA runtime). If the install or
start-up fails on the assigned GPU, set `BACKEND = 'transformers'`, which is slower
(about 1–2 h for 5,000 notes on a T4) but runs anywhere.

In [5]:
BACKEND = 'vllm'   # or 'transformers'
if BACKEND == 'vllm':
    !pip install -q vllm==0.19.0

In [6]:
!python -m src.data.note_synthesis --backend {BACKEND} --seed {SEED} \
    --requests {RAW}/note_requests.jsonl --out {RAW}/notes.jsonl \
    --boilerplate-out {RAW}/boilerplate_notes.jsonl

I0000 00:00:1791755177.769578   14812 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 10-11 21:46:28 [utils.py:233] non-default args: {'dtype': 'half', 'max_model_len': 1024, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 2048, 'max_num_seqs': 128, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 10-11 21:46:29 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 10-11 21:46:29 [model.py:1678] Using max model len 1024
INFO 10-11 21:46:30 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 10-11 21:46:30 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=2048.
Parse safetensors files: 100% 2/2 [00:00<00:00,  4.07it/s]
INFO 10-11 21:46:31 [vllm.py:790] A

### Inspect before building

Read a sample of notes yourself. Record what you checked, and anything you rejected, in
`docs/ai_use_log.md`: the notes are AI-generated research data.

In [7]:
import pandas as pd
from pathlib import Path   # a Path, so a missing file raises FileNotFoundError
notes = pd.read_json(Path(RAW, 'notes.jsonl'), lines=True)
boiler = pd.read_json(Path(RAW, 'boilerplate_notes.jsonl'), lines=True)
print(len(notes), 'notes;', notes['rejected_reason'].notna().sum(), 'rejected:')
print(notes['rejected_reason'].value_counts())
print(len(boiler), 'boilerplate;', boiler['rejected_reason'].notna().sum(), 'rejected')
requests = pd.read_json(Path(RAW, 'note_requests.jsonl'), lines=True)
sample = notes.merge(requests, on='claim_id').sample(8, random_state=SEED)
for row in sample.itertuples():
    print(f'\n[{row.icd11_code}] {row.diagnosis} | {row.patient_age}{row.patient_sex[0]} {row.package}')
    print(row.clinical_notes)

5000 notes; 0 rejected:
Series([], Name: count, dtype: int64)
300 boilerplate; 1 rejected

[5A11] Type 2 diabetes mellitus | 54f outpatient
Reviewed HbA1c: 9.2%. Patient on metformin 1000mg BID, A1C control discussed. Added insulin glargine 10 units OD for better glycemic control. Insulin education and foot examination. Labs: FBS, Lipids. Patient understands importance of diet and exercise.

[JB20.0] Single spontaneous vertex delivery | 33f maternity
Patient underwent a single spontaneous vertex delivery. Normal fetal heart tracing during labor. Complete blood count performed on admission: Hb 10.5 g/dL, WBC 8.5 x10^9/L, platelets 250 x10^9/L. Uterine contractions were good, and there were no signs of infection or abnormal bleeding. Patient stable and discharged in good condition.

[CA02.Z] Acute pharyngitis, unspecified | 43m outpatient
The patient presented with symptoms of sore throat, fever, and mild difficulty swallowing. Examination revealed red and swollen pharyngeal membranes wi

In [8]:
print('Accepted boilerplate:')
for n in boiler.loc[boiler['rejected_reason'].isna(), 'clinical_notes'].sample(10, random_state=SEED):
    print(' -', n)

Accepted boilerplate:
 - Reviewed and managed per protocol. Stable for next review.
 - Patient attended and managed in accordance with standard procedures. Ready for follow-up.
 - Reviewed and managed as per protocols. Stable for discharge.
 - Patient reviewed and managed in accordance with standard protocols. Stable for discharge.
 - Patient evaluated and managed in accordance with standard protocols. Stable for discharge.
 - Patient managed according to standard protocols. Ready for follow-up.
 - Patient reviewed and managed as per protocol. Follow-up advised as needed.
 - Patient reviewed and managed according to protocol. Scheduled for follow-up to monitor progress.
 - Patient reviewed and managed as per protocol. Stable for discharge.
 - Patient evaluated and managed in accordance with standard protocols. Condition stable for further management.


## 4. Stage C: inject fraud, label, split

In [9]:
!python -m src.data.build_dataset --raw {RAW} --out {PROCESSED} --seed {SEED}

{
  "train": {
    "claims": 2700,
    "fraud": 268,
    "inconsistent": 134,
    "text_noise": 810,
    "typologies": {
      "none": 2432,
      "outpatient_to_inpatient": 45,
      "upcoding": 45,
      "anomalous_dependents": 45,
      "ghost_billing": 45,
      "record_fabrication": 44,
      "inflated_caesarean": 44
    }
  },
  "validation": {
    "claims": 900,
    "fraud": 89,
    "inconsistent": 45,
    "text_noise": 270,
    "typologies": {
      "none": 811,
      "record_fabrication": 15,
      "anomalous_dependents": 15,
      "outpatient_to_inpatient": 15,
      "upcoding": 15,
      "ghost_billing": 15,
      "inflated_caesarean": 14
    }
  },
  "test": {
    "claims": 900,
    "fraud": 90,
    "inconsistent": 45,
    "text_noise": 270,
    "typologies": {
      "none": 810,
      "record_fabrication": 15,
      "inflated_caesarean": 15,
      "anomalous_dependents": 15,
      "ghost_billing": 15,
      "outpatient_to_inpatient": 15,
      "upcoding": 15
    }
  },
  "

In [10]:
import json, shutil, datetime
manifest = json.load(open(f'{PROCESSED}/manifest.json'))
manifest['colab_commit'] = COMMIT
json.dump(manifest, open(f'{PROCESSED}/manifest.json', 'w'), indent=2)
stamp = datetime.date.today().isoformat()
!pip freeze > {ENV_LOGS}/requirements_frozen_{stamp}_dataset.txt
pd.DataFrame(manifest['splits']).T[['claims', 'fraud', 'inconsistent', 'text_noise']]

,claims,fraud,inconsistent,text_noise
train,2700,268,134,810
validation,900,89,45,270
test,900,90,45,270
deployment_pool,500,49,25,150
